In [ ]:
import numpy as np
import pandas as pd
from pandas import DataFrame, Series
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.ar_model import AutoReg, ar_select_order
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf, acf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import grangercausalitytests, adfuller
from statsmodels.tsa.api import VAR
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error
import yfinance as yf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression

start_date = '2016-06-01'
tickers = ['^GDAXI', '^STOXX50E']  # dax $ eurostoxx
data = yf.download(tickers, start=start_date, auto_adjust=True)['Close']

data = data.ffill()

target_series = '^GDAXI'
data['target'] = data[target_series].shift(-1)
data = data[:-1]
n_lags = 5

for i in range(1, n_lags + 1):
    data[f'DAX_lag_{i}'] = data['^GDAXI'].shift(i)

for i in range(1, n_lags + 1):
    data[f'EuroStoxx_lag_{i}'] = data['^STOXX50E'].shift(i)

data_clean = data.dropna()

feature_columns = [f'DAX_lag_{i}' for i in range(1, n_lags + 1)] + [f'EuroStoxx_lag_{i}' for i in range(1, n_lags + 1)]
X = data_clean[feature_columns]
y = data_clean['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False, random_state=12)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Линейная регрессия': LinearRegression(),
    'Случайный лес': RandomForestRegressor(random_state=12, n_estimators=100),
    'Градиентный бустинг': GradientBoostingRegressor(random_state=12, n_estimators=100)
}

def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mape_val = mape(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f'{name}: MAE={mae:.2f}, MAPE={mape_val:.2f}%, R²={r2:.2f}')

[*********************100%***********************]  2 of 2 completed


Линейная регрессия: MAE=192.37, MAPE=0.99%, R²=0.99
Случайный лес: MAE=3280.81, MAPE=15.05%, R²=-0.95
Градиентный бустинг: MAE=3311.36, MAPE=15.19%, R²=-0.98


Модель линейной регрессии единственная отвечает критериям адекватности и имеет отличный R^2 (0,95) + наименьшие показатели MAE и MAPE